## 1. Localisation du projet

On détecte automatiquement la racine du repo (le dossier qui contient `src/`), 
pour que le notebook fonctionne peu importe d'où il est lancé, et pour que les
imports depuis `src` fonctionne.

In [7]:
import sys, os
import pandas as pd

# Trouve automatiquement la racine du repo en remontant
# jusqu'à trouver un dossier contenant 'src'
RACINE_PROJET = os.getcwd()
while not os.path.isdir(os.path.join(RACINE_PROJET, "src")):
    parent = os.path.dirname(RACINE_PROJET)
    if parent == RACINE_PROJET:
        raise RuntimeError("Dossier 'src' introuvable — vérifie que tu lances "
                            "le notebook depuis un dossier à l'intérieur du repo")
    RACINE_PROJET = parent

if RACINE_PROJET not in sys.path:
    sys.path.append(RACINE_PROJET)


print("Racine du projet :", RACINE_PROJET)

Racine du projet : e:\cosit-stage-prediction-meteo


## 2. Import des bibliothèques et des modules du projet

On importe les fonctions et constantes partagées définies dans `src/` (chargement des données, préparation des features, création de la cible, et les paramètres communs comme la taille du split et le nombre de folds).

In [8]:
from src.data_loading import charger_donnees
from src.features import preparer_features, creer_cible_pluie
from src.config import FEATURES_COMMUNES, TAILLE_TEST, RANDOM_STATE

## 3. Chargement des données brutes 

On charge le fichier CSV Open-Meteo (en sautant les lignes de métadonnées en en-tête), et on vérifie que le fichier a bien été trouvé et lu.

In [9]:
CHEMIN_CSV = os.path.join(RACINE_PROJET, "data", "data_OPEN_METEO.csv")
print("CSV trouvé :", os.path.exists(CHEMIN_CSV))

df = charger_donnees(CHEMIN_CSV)
df.head()


CSV trouvé : True


,time,weather_code (wmo code),temperature_2m_max (°C),temperature_2m_min (°C),temperature_2m_mean (°C),precipitation_sum (mm),rain_sum (mm),precipitation_hours (h),wind_speed_10m_max (km/h),wind_gusts_10m_max (km/h),wind_direction_10m_dominant (°),relative_humidity_2m_mean (%),surface_pressure_mean (hPa),cloud_cover_mean (%),sunshine_duration (s),mois
0,2015-01-01,2,29.3,24.5,26.9,0.0,0.0,0.0,17.0,34.9,215,79,1010.2,10,41319.21,1
1,2015-01-02,0,29.2,24.6,27.0,0.0,0.0,0.0,15.1,32.0,204,78,1011.0,4,41278.21,1
2,2015-01-03,3,28.9,24.7,26.9,0.0,0.0,0.0,16.7,31.3,174,83,1012.2,39,40985.12,1
3,2015-01-04,3,29.4,23.9,26.5,0.0,0.0,0.0,12.7,24.5,117,73,1012.6,15,41837.27,1
4,2015-01-05,0,29.1,23.0,25.8,0.0,0.0,0.0,12.3,24.1,91,57,1012.2,0,41934.91,1


## 4. Préparation des features 

On renomme les colonnes brutes en noms simples (humidité, pression, vent,...) et on convertit la durée d'ensoleillement de secondes en heures. Cette fonction est commune aux deux modèles du projet.

In [10]:
df = preparer_features(df)
df.head()


,time,weather_code (wmo code),temperature_2m_max (°C),temperature_2m_min (°C),temperature_2m_mean (°C),precipitation_sum (mm),rain_sum (mm),precipitation_hours (h),wind_speed_10m_max (km/h),wind_gusts_10m_max (km/h),...,surface_pressure_mean (hPa),cloud_cover_mean (%),sunshine_duration (s),mois,humidite,pression,vent_vitesse,vent_rafales,nuages,sunshine_duration_heures
0,2015-01-01,2,29.3,24.5,26.9,0.0,0.0,0.0,17.0,34.9,...,1010.2,10,41319.21,1,79,1010.2,17.0,34.9,10,11.477558
1,2015-01-02,0,29.2,24.6,27.0,0.0,0.0,0.0,15.1,32.0,...,1011.0,4,41278.21,1,78,1011.0,15.1,32.0,4,11.466169
2,2015-01-03,3,28.9,24.7,26.9,0.0,0.0,0.0,16.7,31.3,...,1012.2,39,40985.12,1,83,1012.2,16.7,31.3,39,11.384756
3,2015-01-04,3,29.4,23.9,26.5,0.0,0.0,0.0,12.7,24.5,...,1012.6,15,41837.27,1,73,1012.6,12.7,24.5,15,11.621464
4,2015-01-05,0,29.1,23.0,25.8,0.0,0.0,0.0,12.3,24.1,...,1012.2,0,41934.91,1,57,1012.2,12.3,24.1,0,11.648586


## 5. Création de la variable cible

La cible n'existe pas directement dans le CSV : on la crée à partir de `precipitation_sum`. Un jour est considéré "pluvieux" (1) si les précipitations dépassent 1mm, sinon "non pluvieux" (0).

In [13]:
y = creer_cible_pluie(df)

X = df[FEATURES_COMMUNES]

print("Dimension de X :", X.shape)
print("Dimension de y :", y.shape)
print("Proportions de jours pluvieux :", y.mean().round(3))
X.head()

Dimension de X : (3653, 7)
Dimension de y : (3653,)
Proportions de jours pluvieux : 0.508


,humidite,pression,vent_vitesse,vent_rafales,nuages,mois,sunshine_duration_heures
0,79,1010.2,17.0,34.9,10,1,11.477558
1,78,1011.0,15.1,32.0,4,1,11.466169
2,83,1012.2,16.7,31.3,39,1,11.384756
3,73,1012.6,12.7,24.5,15,1,11.621464
4,57,1012.2,12.3,24.1,0,1,11.648586


## 6. Répartition entraînement/test

On sépare les données en un jeu d'entraînement (80%) et un jeu de test (20%), 
mis de côté et non utilisé avant l'utilisation finale. La stratification garantit que la proportion de jours pluvieux reste similaire dans les deux jeux, malgré le déséquilibre des classes.

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TAILLE_TEST,
    random_state=RANDOM_STATE,
    stratify=y  # utile ici car classes déséquilibrées (jours secs vs pluvieux)
)

print("Train :", X_train.shape, "| Test :", X_test.shape)
print("Proportion de pluie (train) :", y_train.mean().round(3))
print("Proportion de pluie (test) :", y_test.mean().round(3))

Train : (2922, 7) | Test : (731, 7)
Proportion de pluie (train) : 0.508
Proportion de pluie (test) : 0.508


## 7. Modèle 1 - Régression logistique (modèle de référence)

On entraîne un modèle simple et interprétable comme point de comparaison pour les modèles suivants (Random Forest, Gradient Boosting). On l'entraîne directement sur le train set, sans recherche d'hyperparamètres complexe, puisque c'est notre référence de base.

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

modele_logit = LogisticRegression(random_state= RANDOM_STATE, max_iter= 1000)
modele_logit.fit(X_train, y_train)

y_pred_logit = modele_logit.predict(X_test)

print("=== Régression logistique ===")
print(classification_report(y_test, y_pred_logit))

=== Régression logistique ===
              precision    recall  f1-score   support

           0       0.70      0.70      0.70       360
           1       0.71      0.71      0.71       371

    accuracy                           0.71       731
   macro avg       0.71      0.71      0.71       731
weighted avg       0.71      0.71      0.71       731



##  Stockage des métriques pour comparaison finale

On garde les métriques de chaque modèle dans un tableau, pour pouvoir comparer les trois modèles côte à côte à la fin, comme demandé pour le raport.

In [16]:
resultats = []

resultats.append({
    "modele": "Régression logistique",
    "precision" : precision_score(y_test, y_pred_logit),
    "rappel" : recall_score(y_test, y_pred_logit),
    "f1" : f1_score(y_test, y_pred_logit)
})

pd.DataFrame(resultats)

,modele,precision,rappel,f1
0,Régression logistique,0.709677,0.71159,0.710633
